# rift `nisar-e2e` — DPS job runner (bounding-box driven)

Discover NISAR **GSLC provisional** granules over an area of interest with `earthaccess`
(CMR), and submit one MAAP DPS job per granule to the `rift-nisar-e2e` OGC process. Each job
runs the full pipeline on the worker: `rift nisar2cog` (GSLC → amplitude COGs) followed by
the **nisar-crevasse** gate + U-Net model (→ `gate_prob.tif` + `unet_prob.tif`). The worker
downloads each granule itself using MAAP/ASF temporary credentials (S3-preferred), so no
pre-staging is needed.

Testing starts on the **`maap-dps-sandbox`** queue (8 GB, 10-min cap) with **`max_tiles=100`**
and **`crop_to_scanned=true`** — the U-Net is CPU-bound, so a capped run keeps the plumbing
test under the sandbox cap. Real full-swath runs use `maap-dps-worker-16gb`/`-32gb` and take
tens of minutes to hours on CPU.

Run this in a **MAAP Hub (OGC) workspace** so maap-py v5 is available.

**Prereqs**
- The `rift-nisar-e2e` OGC process is registered (see `maap/nisar_e2e/README.md`).
- `earthaccess` installed (`pip install earthaccess`) and an Earthdata Login.

In [ ]:
# One-time in a fresh workspace:
# %pip install earthaccess
import earthaccess
import pandas as pd
import datetime, json, os, time
from maap.maap import MAAP

maap = MAAP()

# NISAR provisional collections are auth-gated: you MUST log in to Earthdata or searches
# return 0 results. Run once per workspace.
auth = earthaccess.login(strategy="interactive")

## 1. Area of interest

Thwaites / Pine Island sector. The bounding box `(W, S, E, N)` is set in the search cell
below (§2). Source polygon, for reference:
`POLYGON((-101.9351 -74.7848,-102.4704 -75.4215,-99.852 -75.5511,-99.4242 -74.9087,-101.9351 -74.7848))`

## 2. Search for NISAR GSLC granules (earthaccess / CMR)

NISAR provisional data is discovered via **`earthaccess`** (CMR), not asf-search, and
requires an Earthdata login (done above).

- `short_name="NISAR_L2_GSLC_PROVISIONAL_V1"` — the provisional GSLC collection
- `bounding_box=(W, S, E, N)` — a numeric tuple (not a WKT string)
- `temporal=(start, end)`

Each result exposes `data_links()` (HTTPS) and `data_links(access="direct")` (`s3://`).

In [ ]:
# AOI bounding box (W, S, E, N) from the Thwaites/PIG polygon.
BBOX = (-102.4704, -75.5511, -99.4242, -74.7848)

# Acquisition window (adjust as needed).
START = "2026-07-08T07:00:00Z"
END   = "2026-07-20T06:59:59Z"

results = earthaccess.search_data(
    short_name="NISAR_L2_GSLC_PROVISIONAL_V1",
    bounding_box=BBOX,
    temporal=(START, END),
)
print(f"{len(results)} granules")
len(results)

In [ ]:
# Peek at the first granule's links + id.
if len(results):
    g = results[0]
    print("HTTPS:", g.data_links())
    print("S3   :", g.data_links(access="direct"))
    print("GranuleUR:", g["umm"].get("GranuleUR"))
else:
    print("No granules — check login, widen the date window, or verify the short_name.")

## 3. Build the submit list

The worker downloads the granule itself with MAAP/ASF temporary credentials (S3-preferred),
so we just pass the **`s3://` href** per granule — no pre-staging to a bucket. We collect
`(s3, https)` per result; the submit cell sends `s3_href` and lets the worker fall back to
authenticated HTTPS if needed.

In [ ]:
def first_h5(links):
    return next((u for u in (links or []) if u.lower().endswith(".h5")),
                (links[0] if links else None))

granules = []
for g in results:
    granules.append({
        "name": g["umm"].get("GranuleUR"),
        "https": first_h5(g.data_links()),
        "s3": first_h5(g.data_links(access="direct")),
    })

print(f"{len(granules)} granules")
for gr in granules[:5]:
    print(" ", gr["name"])
    print("     s3   :", gr["s3"])

## 4. Resolve the deployed OGC process id

`maap.list_algorithms()` returns `{"processes": [{"processID": 87, "id": "rift-nisar-e2e", ...}]}`.
**`submit_job` needs the numeric `processID`** (e.g. `87`), *not* the string `id`
(`"rift-nisar-e2e"`) — passing the string (or a stale/deleted id) yields a 500 auth error.
We pick the newest matching `processID`.

In [ ]:
resp = maap.list_algorithms()
procs = resp.json().get("processes", []) if resp.status_code == 200 else []
for p in procs:
    print(p.get("processID"), "|", p.get("id"), "|", p.get("version"))

# submit_job wants the NUMERIC processID (not the string id). Grab the newest matching one.
matches = [p for p in procs
           if "nisar-e2e" in str(p.get("id", "")).lower()
           or "nisar-e2e" in str(p.get("title", "")).lower()]
PROCESS_ID = max((p["processID"] for p in matches), default=None)
print("\nPROCESS_ID =", PROCESS_ID)

## 5. Submit one DPS job per granule

Start with a **single** granule (`granules[:1]`) on `maap-dps-sandbox` to validate the
plumbing. We pass `s3_href` (+ `https_href` fallback) and `access_mode="auto"`; the worker
downloads the granule with MAAP/ASF temporary credentials, geocodes it, and runs the crevasse
model. `submit_job` returns HTTP 202 with `{"id": ..., "status": "accepted"}`.

`MAX_TILES="100"` + `CROP_TO_SCANNED="true"` keep the inference step small and correctly
placed for a quick check. Set `MAX_TILES=""` (no cap) and `CROP_TO_SCANNED="false"` for a
full-swath run on `maap-dps-worker-16gb`/`-32gb`.

In [ ]:
QUEUE = "maap-dps-sandbox"        # testing queue (8 GB, 10-min cap). The U-Net is
                                   # CPU-bound; max_tiles=100 keeps a capped run under the
                                   # cap. Use maap-dps-worker-16gb / -32gb (and MAX_TILES="")
                                   # for a real full-swath run.
TAG = "nisar-e2e-thwaites"
POLS = ""                          # empty = all freq-A pols
AMP_ONLY = "false"
MAX_TILES = "100"                  # cap inference tiles for the sandbox test; "" = full swath
CROP_TO_SCANNED = "true"           # crop the ML GeoTIFFs to the scanned bbox (pairs w/ max_tiles)

SUBMIT = granules[:1]              # <-- widen to `granules` once the first job succeeds
assert PROCESS_ID, "PROCESS_ID not resolved — is the process deployed? (see §4)"
assert SUBMIT, "No granules — check the search cells."

rows = []
for i, gr in enumerate(SUBMIT, start=1):
    # Always submit the exact granule (its s3/https hrefs from the §2 AOI search).
    inputs = {
        "access_mode": "auto",     # S3 first, authenticated-HTTPS fallback
        "s3_href": gr["s3"],
        "https_href": gr["https"],
        "pols": POLS,
        "amp_only": AMP_ONLY,
        "max_tiles": MAX_TILES,
        "crop_to_scanned": CROP_TO_SCANNED,
    }
    r = maap.submit_job(process_id=PROCESS_ID, inputs=inputs,   # PROCESS_ID is the int, e.g. 87
                        queue=QUEUE, dedup=True, tag=TAG)
    body = r.json() if r.status_code == 202 else {}
    job_id, status = body.get("jobID") or body.get("id"), body.get("status", r.text)
    print(f"[{i}/{len(SUBMIT)}] {r.status_code} job_id={job_id} status={status}")
    rows.append({"n": i, "granule": gr["name"], "s3_href": gr["s3"], "job_id": job_id,
                 "submit_status": status, "http": r.status_code,
                 "submit_time": datetime.datetime.now().isoformat()})

submit_df = pd.DataFrame(rows)
out_dir = os.path.expanduser("~/my-public-bucket/dps_submission_results")
os.makedirs(out_dir, exist_ok=True)
stamp = datetime.datetime.now().strftime("%Y%m%d%H%M")
csv_path = f"{out_dir}/nisar-e2e_{TAG}_{stamp}.csv"
submit_df.to_csv(csv_path, index=False)
print("saved", csv_path)
submit_df

## 6. Monitor jobs and fetch results

`get_job_status` / `get_job_result` return JSON in maap-py v5. Results land under
`~/my-private-bucket/dps_output/rift-nisar-e2e/...` — the amplitude/phase COGs plus
`gate_prob.tif` + `unet_prob.tif`.

In [ ]:
for job_id in [j for j in submit_df["job_id"].tolist() if j]:
    s = maap.get_job_status(job_id)
    st = s.json().get("status") if s.status_code == 200 else s.text
    print(job_id, "->", st)

In [ ]:
# Once a job shows succeeded, inspect its outputs:
SUCCESS_JOB_ID = ""  # paste a job id
if SUCCESS_JOB_ID:
    r = maap.get_job_result(SUCCESS_JOB_ID)
    print(json.dumps(r.json(), indent=2) if r.status_code == 200 else r.text)
    m = maap.get_job_metrics(SUCCESS_JOB_ID)
    if m.status_code == 200:
        print(json.dumps(m.json(), indent=2))